# 🎯 BỘ CÔNG CỤ KIỂM CHỨNG GIẢ THUYẾT KHOA HỌC KHÓA LUẬN TỐT NGHIỆP 2026
## Hậu kiểm thực nghiệm Đa Tập Dữ liệu (COCOA - D2SA - KINS) trên Google Colab T4 / A100

> **Đề tài:** Phân đoạn Thực thể Amodal với Kiến trúc SG-GBA Swin-UNet  
> **Người thực hiện:** Võ Thế Nguyễn & Phan Quang Nghĩa  
> **Mục đích khoa học:** Thu thập toàn bộ bằng chứng định lượng (Quantitative Proofs) cho **Chương 4 Khóa luận** mà **KHÔNG CẦN train lại mô hình**:  
> 1. **Gate Intervention Test:** Đánh giá nhân quả cổng $\alpha$ (mặc định vs. $\alpha = 0.0$ vs. $\alpha = -\alpha$).  
> 2. **Class Frequency / Long-Tail Analysis:** Phân rã Occ. mIoU theo 3 nhóm Head, Medium, Tail giải thích hiện tượng FiLM 444K tham số.  
> 3. **Occlusion Severity Breakdown:** Đánh giá độ bền bỉ khi tỷ lệ che khuất $r_{\text{occ}}$ tăng từ Low ($\le 20\%$) lên Heavy ($> 50\%$).  
> 4. **Spatial Attention Paradox Proof:** Tính toán chỉ số $\text{OAR} = \frac{\text{Mean Act}(M_{occ})}{\text{Mean Act}(M_{vis})}$ so sánh Bottleneck ($7\times 7$) và Decoder ($224\times 224$).  
> 5. **Đồ thị xuất bản 300 DPI:** Xuất 3 biểu đồ khoa học và báo cáo LaTeX Table tự động lưu vào Google Drive.

In [ ]:
# 1. KIỂM TRA PHẦN CỨNG GPU (T4 / L4 / A100)
!nvidia-smi
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🚀 PyTorch Version: {torch.__version__} | Device: {device}')
if torch.cuda.is_available():
    print(f'✨ GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'💾 VRAM Dung lượng: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


### 🔹 BƯỚC 1: Mount Google Drive, Clone Repo (Nhánh `new_framework`) & Nối Symlink Lưu Trữ Bền Vững
Tự động quét thư mục `amodal_shape_prediction` trên Google Drive (hỗ trợ cả tài khoản cá nhân, lối tắt Shortcut, và Shared Drive).

In [ ]:
from google.colab import drive
import os, sys

# 1. Kết nối Google Drive
drive.mount('/content/drive')

# 2. Tự động nhận diện thư mục dự án trên Drive
possible_drive_dirs = [
    '/content/drive/MyDrive/amodal_shape_prediction',
    '/content/drive/MyDrive/KLTN_Amodal_2026',
    '/content/drive/MyDrive/CV+/data',
    '/content/drive/Shareddrives/amodal_shape_prediction'
]
DRIVE_DIR = None
for p in possible_drive_dirs:
    if os.path.exists(p):
        DRIVE_DIR = p
        break

if DRIVE_DIR is None:
    for item in os.listdir('/content/drive/MyDrive'):
        full_p = os.path.join('/content/drive/MyDrive', item)
        if os.path.isdir(full_p) and ('amodal' in item.lower() or 'kltn' in item.lower()):
            DRIVE_DIR = full_p
            break

if DRIVE_DIR is None:
    DRIVE_DIR = '/content/drive/MyDrive/amodal_shape_prediction'
    os.makedirs(DRIVE_DIR, exist_ok=True)

print(f'📂 Thư mục Google Drive được chọn: {DRIVE_DIR}')

# 3. Clone hoặc pull nhánh new_framework vào SSD Colab
%cd /content
REPO_DIR = '/content/amodal_shape_prediction'
if not os.path.exists(REPO_DIR):
    !git clone -b new_framework https://github.com/vothenguyen/amodal_shape_prediction.git
else:
    %cd {REPO_DIR}
    !git fetch origin && git checkout new_framework && git pull origin new_framework

%cd {REPO_DIR}

# 4. Tạo thư mục bền vững trên Drive và liên kết symlink
for sub in ['checkpoints', 'logs', 'results', 'results/diagnostics']:
    os.makedirs(os.path.join(DRIVE_DIR, sub), exist_ok=True)

!rm -rf checkpoints logs results
!ln -sf "{DRIVE_DIR}/checkpoints" checkpoints
!ln -sf "{DRIVE_DIR}/logs" logs
!ln -sf "{DRIVE_DIR}/results" results

# 5. Cài đặt các thư viện phụ thuộc
!pip install -q timm albumentations pycocotools tqdm kagglehub matplotlib scipy pandas
print('✅ Môi trường Colab & Google Drive đã đồng bộ hoàn tất!')


### 🔹 BƯỚC 2: Chuẩn bị Dữ liệu Validation Siêu Tốc (COCOA - D2SA - KINS)
- **COCOA:** Giải nén `annotations.zip` từ Drive + chỉ tải `val2014.zip` (~6GB trong ~1 phút) thay vì tải `train2014` 13GB không cần thiết!
- **D2SA:** Giải nén `d2s_amodal_annotations_v1.tar.xz` và ảnh từ Drive vào Colab SSD.
- **KINS:** Tải tự động qua KaggleHub (`kitti-2d-object-detection` và `kins-data`).

In [ ]:
import os, zipfile, tarfile, shutil
%cd /content/amodal_shape_prediction

print('📦 BẮT ĐẦU CHUẨN BỊ DỮ LIỆU ĐÁNH GIÁ (VAL SET)...')

# 1. COCOA
print('\n1️⃣ Chuẩn bị COCOA...')
cocoa_ann_zip_candidates = [
    os.path.join(DRIVE_DIR, 'data', 'COCOA', 'annotations.zip'),
    os.path.join(DRIVE_DIR, 'data', 'annotations.zip'),
    '/content/drive/MyDrive/CV+/data/annotations.zip'
]
cocoa_ann_zip = None
for cand in cocoa_ann_zip_candidates:
    if os.path.exists(cand):
        cocoa_ann_zip = cand
        break

os.makedirs('data/COCOA', exist_ok=True)
if cocoa_ann_zip:
    print(f'   ↳ Giải nén annotations từ: {cocoa_ann_zip}')
    !unzip -qo "{cocoa_ann_zip}" -d data/COCOA/
else:
    print('   ⚠️ Không tìm thấy annotations.zip trên Drive, kiểm tra dữ liệu local...')

os.makedirs('data', exist_ok=True)
if not os.path.exists('data/val2014'):
    print('   ↳ Đang tải siêu tốc val2014.zip từ cocodataset.org...')
    !wget -q -c http://images.cocodataset.org/zips/val2014.zip -O data/val2014.zip
    !unzip -qo data/val2014.zip -d data/ && rm -f data/val2014.zip
    print('   ✅ Đã giải nén val2014!')
else:
    print('   ✅ data/val2014 đã tồn tại sẵn!')

# 2. D2SA
print('\n2️⃣ Chuẩn bị D2SA...')
os.makedirs('data/D2SA', exist_ok=True)
d2sa_drive_data = os.path.join(DRIVE_DIR, 'data', 'D2SA')
for df in ['d2s_amodal_annotations_v1.tar.xz', 'd2s_amodal_images_v1.tar.xz', 'D2SA.zip']:
    full_df = os.path.join(d2sa_drive_data, df)
    if os.path.exists(full_df):
        print(f'   ↳ Đang giải nén {df} từ Drive...')
        if df.endswith('.tar.xz'):
            !tar -xf "{full_df}" -C data/D2SA/
        elif df.endswith('.zip'):
            !unzip -qo "{full_df}" -d data/D2SA/

if not os.path.exists('data/D2SA/images') and os.path.exists(os.path.join(d2sa_drive_data, 'images')):
    !ln -sf "{d2sa_drive_data}/images" data/D2SA/images
    !ln -sf "{d2sa_drive_data}"/*.json data/D2SA/

# 3. KINS
print('\n3️⃣ Chuẩn bị KINS (KaggleHub)...')
import kagglehub
kitti_path = kagglehub.dataset_download('anupammajhi/kitti-2d-object-detection')
kins_path = kagglehub.dataset_download('ryanthenguyen/kins-data')

!rm -rf /content/kitti_data /content/kins_data
os.symlink(f'{kitti_path}/data_object_image_2', '/content/kitti_data')
os.symlink(kins_path, '/content/kins_data')
print('   ✅ KINS data sẵn sàng tại /content/kitti_data và /content/kins_data!')
print('\n🎯 TOÀN BỘ 3 TẬP DỮ LIỆU ĐÃ SẴN SÀNG!')


### 🔹 BƯỚC 3: Chạy Bộ Chẩn đoán Thực nghiệm Hậu kiểm (`run_thesis_diagnostics.py`)
Thực hiện toàn bộ 4 thí nghiệm khoa học cho cả 3 tập dữ liệu (COCOA, D2SA, KINS):
- `--dataset all` (hoặc chọn riêng `cocoa`, `d2sa`, `kins`)
- `--batch-size 64` (tối ưu thông lượng GPU Colab)
- `--num-workers 4`

In [ ]:
%cd /content/amodal_shape_prediction
!python scripts/run_thesis_diagnostics.py \
    --dataset all \
    --batch-size 64 \
    --num-workers 4 \
    --drive-dir "{DRIVE_DIR}" \
    --output-dir results/diagnostics


### 🔹 BƯỚC 4: Hiển thị Báo cáo Tổng kết & 3 Biểu đồ Khoa học 300 DPI
Hiển thị trực tiếp kết quả và hình ảnh trực quan trên màn hình Colab để kiểm tra ngay lập tức.

In [ ]:
import os, json
from IPython.display import Image, display, Markdown

report_path = 'results/diagnostics/THESIS_DIAGNOSTIC_REPORT.md'
if os.path.exists(report_path):
    with open(report_path, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))

print('\n' + '='*70)
print('📈 1. BIỂU ĐỒ QUỸ ĐẠO TIẾN HÓA CỔNG HỌC ĐƯỢC ALPHA(t) QUA 30 EPOCHS:')
print('='*70)
if os.path.exists('results/diagnostics/fig1_gate_alpha_trajectory.png'):
    display(Image(filename='results/diagnostics/fig1_gate_alpha_trajectory.png', width=950))

print('\n' + '='*70)
print('📈 2. BIỂU ĐỒ PHÂN RÃ THEO MỨC ĐỘ CHE KHUẤT VÀ TẦN SUẤT LỚP (LONG-TAIL):')
print('='*70)
if os.path.exists('results/diagnostics/fig2_occlusion_severity_and_head_tail.png'):
    display(Image(filename='results/diagnostics/fig2_occlusion_severity_and_head_tail.png', width=950))

print('\n' + '='*70)
print('📈 3. ĐỊNH LƯỢNG NGHỊCH LÝ CHÚ Ý (SPATIAL ATTENTION PARADOX & OAR):')
print('='*70)
if os.path.exists('results/diagnostics/fig3_oar_attention_paradox.png'):
    display(Image(filename='results/diagnostics/fig3_oar_attention_paradox.png', width=950))

print('\n🎉 TOÀN BỘ KẾT QUẢ ĐÃ ĐƯỢC TỰ ĐỘNG SAO LƯU VÀO GOOGLE DRIVE TẠI:')
print(f'👉 {DRIVE_DIR}/results/diagnostics/')
